# 5 · unspecified-goal kv-transfer — algorithm swap with an un-named sub-goal

The usual kv-transfer specifies every sub-goal in the task description. Here the instruction names **only the algorithm**
(`Implement breadth-first search …`) and the function is called `func`. The sub-goal at the pivot (e.g. *make a queue* vs
*make a stack*) is never written down; the model has to derive it from the algorithm's name.

```
p_src: "Implement A"  + shared prefill  ->  sub-goal of A      (e.g. BFS -> queue)
p_dst: "Implement B"  + shared prefill  ->  sub-goal of B      (e.g. DFS -> stack)
```
**Intervention** = path patching (`src.diff.path_patch.intervention_stack`, the batch eval's code), I1 off:

* **I3, pivot-onwards KV transfer** (always on): the goal-250 heads read SRC's k/v over the task span **only from the last
  query on** (the pivot on the prefill pass, then each generated token), so every pre-pivot DST latent is untouched.
  Task span (`SPAN`): `"name+signature"` = the algorithm name in the instruction + the `def func(...):` line of the prefill
  (default) · `"name"` · `"instruction"` (the whole user turn). The name span is `\n`-padded to equal length (`pad`); the
  signature is token-identical in both prompts. Values ×`V_SCALE` (keys / attention pattern unchanged), OV boost `ALPHA` at -1.
* **I2, pre-commitment-swap residual transfer** (optional, `I2`): every layer's output residual over `<|im_end|> … pivot-1`
  is replaced with **SRC's** (the pairs have no third algorithm to act as p_res, so p_src is the donor — the Appendix J
  pre-commitment swap).

**Readouts** — GoalScope at the pivot (last prompt token) plus any prompt positions in `READ_AT`, on clean SRC / DST /
patched DST. Under I3 only the last query reads SRC's span, so patched readouts at earlier positions differ from clean DST
only through I2.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")   # check nvidia-smi first (shared machine)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, re
from contextlib import ExitStack
import torch
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
from src.diff.functional import (align_kv_spans, boost_head_ov, cache_src_proj, get_prompt, greedy,
                                 headwise_kv_patch, heads_per_layer, token_span)
from src.goals.goalscope import apply_goalscope
from src.tokens import prepare_input
from src.utils import env_utils

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich import print as rich_print
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]
print(f"{len(GOAL_HEADS_250)} goal heads")

2026-10-18 15:08:50 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
250 goal heads


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
SPAN = "name+signature"     # I3 task span: "name+signature" (algorithm name + `def func(...):` line) | "name" | "instruction"
V_SCALE = 4.0               # I3: x the transplanted SRC values (attention pattern unchanged); 1.0 = off
ALPHA = 1.0                 # I3: OV-write boost of PATCH_HEADS at -1 (pivot / each generated token); 1.0 = off
I2 = False                  # I2: pre-commitment swap — DST's <|im_end|> .. pivot-1 residuals (every layer) <- SRC's
ALIGN_KNOB = "pad"          # "pad" = newline-pad the shorter name span so the dest span is fully overwritten | "right"
PATCH_HEADS = GOAL_HEADS_250
READOUT_HEADS = GOAL_HEADS_250
MAX_NEW = 64                # greedy tokens per generation
RO_MAX_NEW = 48             # tokens per GoalScope verbalization
#------------------------------------------------------------------------------------------------------
from types import SimpleNamespace
import src.diff.path_patch as PP
PP_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=READOUT_HEADS, gsp=getattr(P, "gsp", None))

## Contrastive pairs — pick one

Each pair: two algorithms with the same signature and a shared code prefix that is natural for both; the pivot is the
last prefill token (the indentation token before the first diverging line — Qwen tokenizes `\n    x` as `\n` … `'   '`, `' x'`).
`kw` regexes give a quick verdict on the first line the model writes. `FLIP=True` swaps SRC and DST.

In [12]:
INSTRUCTION = "Implement {name} in Python. Name the function `func`."

PAIRS = {
    "bfs_dfs": dict(
        a=dict(id="bfs", name="breadth-first search graph traversal", kw=r"queue|deque|popleft"),
        b=dict(id="dfs", name="iterative depth-first search graph traversal", kw=r"stack"),
        prefill="<code>\ndef func(graph, start):\n    visited = set()\n   ",
    ),
    "bsearch_leftmost": dict(
        a=dict(id="binary_search", name="binary search", kw=r"==\s*target"),
        b=dict(id="leftmost", name="binary search that returns the index of the leftmost occurrence of target",
               kw=r"<\s*target|result|ans|res\b"),
        prefill=("<code>\ndef func(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n"
                 "        mid = (lo + hi) // 2\n       "),
    ),
    "merge_quick": dict(
        a=dict(id="merge_sort", name="merge sort", kw=r"mid|merge|//\s*2"),
        b=dict(id="quick_sort", name="quick sort", kw=r"pivot"),
        prefill="<code>\ndef func(arr):\n    if len(arr) <= 1:\n        return arr\n   ",
    ),
    # ── added 2026-10-18; clean py first lines checked on Coder-32B (greedy): every side hits only its own kw ──
    "dijkstra_bfs": dict(
        a=dict(id="dijkstra", name="Dijkstra's shortest path algorithm from a start node", kw=r"heap|priority|\(\s*0\s*,"),
        b=dict(id="bfs_shortest", name="unweighted shortest path from a start node using breadth-first search",
               kw=r"deque|\[start\]"),
        prefill="<code>\ndef func(graph, start):\n    dist = {start: 0}\n   ",
    ),
    "prim_kruskal": dict(
        a=dict(id="prim", name="Prim's minimum spanning tree algorithm", kw=r"heap|adj|graph|visited|defaultdict"),
        b=dict(id="kruskal", name="Kruskal's minimum spanning tree algorithm", kw=r"sort|parent|find|rank"),
        prefill="<code>\ndef func(n, edges):\n    mst = []\n   ",
    ),
    "bubble_selection": dict(
        a=dict(id="bubble_sort", name="bubble sort", kw=r"n\s*-\s*i\s*-\s*1|\+\s*1\]"),
        b=dict(id="selection_sort", name="selection sort", kw=r"min"),
        prefill="<code>\ndef func(arr):\n    n = len(arr)\n    for i in range(n):\n       ",
    ),
    "fib_rec_iter": dict(
        a=dict(id="fib_recursive", name="the n-th Fibonacci number using naive recursion", kw=r"func\("),
        b=dict(id="fib_iterative", name="the n-th Fibonacci number iteratively with two running variables",
               kw=r"a,\s*b|prev|curr"),
        prefill="<code>\ndef func(n):\n    if n < 2:\n        return n\n   ",
    ),
    "power_fast_naive": dict(
        a=dict(id="fast_power", name="exponentiation by squaring", kw=r"while"),
        b=dict(id="naive_power", name="integer power by repeated multiplication in a loop", kw=r"for"),
        prefill="<code>\ndef func(base, exp):\n    result = 1\n   ",
    ),
    "twosum_hash_pointers": dict(
        a=dict(id="twosum_hash", name="two-sum that returns the indices of the two numbers, using a hash map",
               kw=r"seen|\{\}|dict|map"),
        b=dict(id="twosum_pointers",
               name="two-sum on a sorted array that returns the indices of the two numbers, using two pointers",
               kw=r"left|right|\blo\b|\bhi\b|\bi\s*,\s*j\b"),
        prefill="<code>\ndef func(nums, target):\n   ",
    ),
    "palindrome_reverse_pointers": dict(
        a=dict(id="pal_reverse", name="a palindrome check that compares the string with its reverse", kw=r"\[::-1\]"),
        b=dict(id="pal_pointers", name="a palindrome check that walks two pointers inward from both ends",
               kw=r"left|right|\blo\b|\bhi\b|\bi\s*,\s*j\b"),
        prefill="<code>\ndef func(s):\n   ",
    ),
    "toposort_kahn_dfs": dict(
        a=dict(id="kahn", name="topological sort using Kahn's algorithm", kw=r"in_?degree"),
        b=dict(id="topo_dfs", name="topological sort using depth-first search", kw=r"visit"),
        prefill="<code>\ndef func(graph):\n    result = []\n   ",
    ),
    "lcs_dp_memo": dict(
        a=dict(id="lcs_bottom_up", name="longest common subsequence length with bottom-up dynamic programming",
               kw=r"dp|table"),
        b=dict(id="lcs_top_down", name="longest common subsequence length with top-down memoized recursion",
               kw=r"memo|cache|def "),
        prefill="<code>\ndef func(a, b):\n    m, n = len(a), len(b)\n   ",
    ),
}

PAIR, FLIP = "merge_quick", True      # <- choose
# GoalScope positions besides the pivot (always included). Inside the prompt only.
#   int < 0  -> offset from the end of the prompt (-1 = pivot)      int >= 0 -> absolute index in the DST tokenization
#   str      -> last token in the prompt whose text (stripped) equals it, e.g. " visited" (skipped if absent)
READ_AT = [-2, " visited", " func"]

## Build SRC / DST prompts — task span (yellow), transplanted k/v window (orange), pivot (salmon)

In [13]:
def question(alg):
    return INSTRUCTION.format(name=alg["name"])


def span_text(alg):
    return question(alg) if SPAN == "instruction" else alg["name"]


def signature_line(prefill):
    return next(ln for ln in prefill.split("\n") if ln.lstrip().startswith("def "))


def build(pair_key, flip=False):
    """Prompts + the I3 window (src_w / dst_w, paired 1:1) + the I2 window (seg on DST, res_seg on SRC), as the
    `g` dict PP.intervention_stack expects (I2 donor = SRC: res_tok = src_tok)."""
    pair = PAIRS[pair_key]
    src, dst = (pair["b"], pair["a"]) if flip else (pair["a"], pair["b"])
    src_prompt, dst_prompt = get_prompt(question(src), pair["prefill"]), get_prompt(question(dst), pair["prefill"])
    s0, t0 = token_span(src_prompt, span_text(src)), token_span(dst_prompt, span_text(dst))
    src_tok, dst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
        mt, src_prompt, dst_prompt, s0, t0, mode=ALIGN_KNOB)
    src_w, dst_w = list(src_pos), list(dst_pos)
    s_ids, d_ids = src_tok["input_ids"][0].tolist(), dst_tok["input_ids"][0].tolist()
    L_s, L_d = len(s_ids), len(d_ids)
    if SPAN == "name+signature":   # + the `def func(...):` line (token-identical; located in DST, aligned from the end)
        sig = signature_line(pair["prefill"])
        lo, hi = token_span(dst_prompt, sig)
        shift = L_d - prepare_input(prompts=dst_prompt, tokenizer=mt)["input_ids"].shape[1]   # pad tokens before it
        sig_d = list(range(lo + shift, hi + shift))
        sig_s = [q - L_d + L_s for q in sig_d]
        assert [s_ids[q] for q in sig_s] == [d_ids[q] for q in sig_d], "signature tokens differ"
        src_w, dst_w = src_w + sig_s, dst_w + sig_d
    # I2 window: last <|im_end|> .. pivot-1, token-identical in both prompts (aligned from the end)
    im_end = mt.tokenizer.convert_tokens_to_ids("<|im_end|>")
    seg = list(range(max(i for i, t in enumerate(d_ids) if t == im_end), L_d - 1))
    res_seg = [q - L_d + L_s for q in seg]
    assert [s_ids[q] for q in res_seg] == [d_ids[q] for q in seg], "SRC/DST progress tokens differ"
    return dict(pair=pair_key, flip=flip, src=src, dst=dst, src_tok=src_tok, dst_tok=dst_tok,
                src_pos=src_w, dst_pos=dst_w, src_span=src_span, dst_span=dst_span,
                src_w=src_w, dst_w=dst_w, seg=seg, res_tok=src_tok, res_seg=res_seg)


def show_tokens(tok, title, task_span, kv_pos):
    ids = tok["input_ids"][0].tolist()
    task, kv, piv = set(range(*task_span)), set(kv_pos), len(ids) - 1
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid])
        style = ("black on salmon1" if i == piv else "black on dark_orange" if i in kv
                 else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{title}[/bold]  L={len(ids)}  pivot={mt.tokenizer.decode([ids[-1]])!r}",
                     title_align="left", border_style="grey42"))


def resolve_positions(tok, read_at):
    """READ_AT -> sorted absolute positions in `tok` (pivot always included)."""
    ids = tok["input_ids"][0].tolist()
    n, out = len(ids), {len(ids) - 1}
    for p in read_at:
        if isinstance(p, int):
            out.add(n + p if p < 0 else p)
        else:
            hits = [i for i, t in enumerate(ids) if mt.tokenizer.decode([t]).strip() == p.strip()]
            if not hits:
                print(f"READ_AT: token {p!r} not in this prompt — skipped")
                continue
            out.add(hits[-1])
    assert all(0 <= q < n for q in out), f"positions out of range: {sorted(out)} (L={n})"
    return sorted(out)


B = build(PAIR, FLIP)
for name, tok, span, pos in [("SRC", B["src_tok"], B["src_span"], B["src_pos"]),
                             ("DST", B["dst_tok"], B["dst_span"], B["dst_pos"])]:
    alg = B["src"] if name == "SRC" else B["dst"]
    show_tokens(tok, f"{name} · {alg['id']} · span={SPAN} · I3 window (orange) {len(pos)} tok", span, pos)
print(f"SRC = {B['src']['id']}   DST = {B['dst']['id']}   (I3: SRC span k/v -> DST, last-onwards"
      f"{' · I2: DST progress <- SRC ' + str((B['seg'][0], B['seg'][-1])) if I2 else ''})")

╭─ SRC · quick_sort · span=name+signature · I3 window (orange) 6 tok  L=78  pivot='   ' ──────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Implement quick sort in Python. Name the function `func`.<|im_end|>                                             │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(arr):                                                                                                  │
│     if len(arr) <= 1:                                                                                           │
│         return arr                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · merge_sort · span=name+signature · I3 window (orange) 6 tok  L=78  pivot='   ' ──────────────────────────╮
│ <|im_start|>system                                                                                              │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>                                                              │
│ <|im_start|>user                                                                                                │
│ Implement merge sort in Python. Name the function `func`.<|im_end|>                                             │
│ <|im_start|>assistant                                                                                           │
│ <code>                                                                                                          │
│ def func(arr):                                                                                                  │
│     if len(arr) <= 1:                                                                                           │
│         return arr                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC = quick_sort   DST = merge_sort   (I3: SRC span k/v -> DST, last-onwards)


### Token picker — the prompt tail with absolute index and offset from end (for `READ_AT`)

In [14]:
def token_picker(tok, tail=40):
    ids = tok["input_ids"][0].tolist()
    n = len(ids)
    t = Table(show_header=True, header_style="bold", box=None, pad_edge=False)
    for c in ("abs", "from end", "token"):
        t.add_column(c, justify="right" if c != "token" else "left")
    for i in range(max(0, n - tail), n):
        t.add_row(str(i), str(i - n), repr(mt.tokenizer.decode([ids[i]])))
    rich_print(t)


token_picker(B["dst_tok"])
print("READ_AT ->", resolve_positions(B["dst_tok"], READ_AT))

abs  from end  token         
 38       -40  'user'        
 39       -39  '\n'          
 40       -38  'Implement'   
 41       -37  ' merge'      
 42       -36  ' sort'       
 43       -35  ' in'         
 44       -34  ' Python'     
 45       -33  '.'           
 46       -32  ' Name'       
 47       -31  ' the'        
 48       -30  ' function'   
 49       -29  ' `'          
 50       -28  'func'        
 51       -27  '`.'          
 52       -26  '<|im_end|>'  
 53       -25  '\n'          
 54       -24  '<|im_start|>'
 55       -23  'assistant'   
 56       -22  '\n'          
 57       -21  '<'           
 58       -20  'code'        
 59       -19  '>\n'         
 60       -18  'def'         
 61       -17  ' func'       
 62       -16  '(arr'        
 63       -15  '):\n'        
 64       -14  '   '         
 65       -13  ' if'         
 66       -12  ' len'        
 67       -11  '(arr'        
 68       -10  ')'           
 69        -9  ' <='         
 70        -8  ' '           
 71        -7  '1'           
 72        -6  ':\n'         
 73        -5  '       '     
 74        -4  ' return'     
 75        -3  ' arr'        
 76        -2  '\n'          
 77        -1  '   '

READ_AT: token ' visited' not in this prompt — skipped
READ_AT -> [61, 76, 77]


## Intervention — generations + GoalScope readouts

In [15]:
def verdict(gen, b):
    first = next((ln for ln in gen.split("\n") if ln.strip()), "")
    hs, hd = bool(re.search(b["src"]["kw"], first)), bool(re.search(b["dst"]["kw"], first))
    return "SRC" if hs and not hd else "DST" if hd and not hs else "both" if hs else "neither"


def run(b, read_at=READ_AT):
    src_tok, dst_tok = b["src_tok"], b["dst_tok"]
    # I3 (last-onwards) [+ I2] — the path-patching batch eval's intervention stack (fresh single-use stack per forward)
    patched = PP.intervention_stack(b, task_heads=PATCH_HEADS, goal_heads=PATCH_HEADS, i1=False, i2=I2, i3=True,
                                    i3_query="last", v_scale=V_SCALE, alpha=ALPHA, donor="res", ctx=PP_CTX)

    def patched_iface(mt_, inputs, patches=None, model_kwargs=None):
        with patched():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))

    # generations: I3 reads SRC's span at the last query of every forward = pivot on prefill, then each generated token
    gens = dict(src=greedy(src_tok, max_new=MAX_NEW), dst=greedy(dst_tok, max_new=MAX_NEW))
    with patched():
        gens["patched"] = greedy(dst_tok, max_new=MAX_NEW)

    # readout positions: resolved on DST, mapped to SRC by offset from the end (identical under pad)
    n_s, n_d = src_tok["input_ids"].shape[1], dst_tok["input_ids"].shape[1]
    pos_d = resolve_positions(dst_tok, read_at)
    pos_s = [p - n_d + n_s for p in pos_d]
    for ps, pd in zip(pos_s, pos_d):
        ts, td = (mt.tokenizer.decode([t["input_ids"][0, q]]) for t, q in ((src_tok, ps), (dst_tok, pd)))
        if ts != td:
            print(f"warning: SRC pos {ps} {ts!r} != DST pos {pd} {td!r} (position is inside the differing span?)")

    def ro(tok, pos, iface=None):
        res = apply_goalscope(prompt=tok, mt=mt, selected_heads=READOUT_HEADS, strategy="additive_norm",
                              alpha=0.1, steer_layer_range=(35, 46), batch_size=len(pos),
                              max_new_tokens=RO_MAX_NEW, goalscope_prompt=getattr(P, "gsp", None),
                              token_positions=pos, include_noncode=False, show=False, patch_interface=iface)
        return {r["pos"]: r["patched_gen"] for r in res}

    ros = dict(src=ro(src_tok, pos_s), dst=ro(dst_tok, pos_d), patched=ro(dst_tok, pos_d, iface=patched_iface))
    rows = [dict(pos=pd, from_end=pd - n_d, token=mt.tokenizer.decode([dst_tok["input_ids"][0, pd]]),
                 src=ros["src"][ps], dst=ros["dst"][pd], patched=ros["patched"][pd])
            for ps, pd in zip(pos_s, pos_d)]
    return dict(gens=gens, verdicts={k: verdict(g, b) for k, g in gens.items()}, rows=rows)


def show(b, out):
    head = (f"{b['pair']}{' (flipped)' if b['flip'] else ''} · SRC={b['src']['id']} -> DST={b['dst']['id']} · "
            f"I3 last-onwards · span={SPAN} · V×{V_SCALE:g} · α={ALPHA} · {ALIGN_KNOB} · I2={'on (SRC donor)' if I2 else 'off'}")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        rich_print(Panel(Text(out["gens"][k]), title=f"[{col}]{k}[/{col}]  verdict={out['verdicts'][k]}",
                         title_align="left", border_style="grey42"))
    t = Table(title=head, show_lines=True)
    t.add_column("pos", justify="right"); t.add_column("from end", justify="right"); t.add_column("token")
    for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
        t.add_column(Text(f"{'clean SRC' if k == 'src' else 'DST' if k == 'dst' else 'patched DST'}", style=col), ratio=1)
    for r in out["rows"]:
        tok = Text(repr(r["token"]), style="bold salmon1" if r["from_end"] == -1 else "")
        t.add_row(str(r["pos"]), str(r["from_end"]), tok, *(Text(r[k].strip()) for k in ("src", "dst", "patched")))
    Console(width=200).print(t)


OUT = run(B)
show(B, OUT)

READ_AT: token ' visited' not in this prompt — skipped


GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

╭─ src  verdict=both ─────────────────────────────────────────────────────────────────────────────────────────────╮
│  pivot = arr[len(arr) // 2]                                                                                     │
│     left = [x for x in arr if x < pivot]                                                                        │
│     middle = [x for x in arr if x == pivot]                                                                     │
│     right = [x for x in arr if x > pivot]                                                                       │
│     return func(left) + middle + func(right)                                                                    │
│ </                                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ dst  verdict=DST ──────────────────────────────────────────────────────────────────────────────────────────────╮
│  mid = len(arr) // 2                                                                                            │
│     left_half = func(arr[:mid])                                                                                 │
│     right_half = func(arr[mid:])                                                                                │
│     return merge(left_half, right_half)                                                                         │
│                                                                                                                 │
│ def merge(left, right):                                                                                         │
│     sorted_array = []                                                                                           │
│     left_index, right_index = 0, 0                                                                              │
│     while left_index <                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched  verdict=both ─────────────────────────────────────────────────────────────────────────────────────────╮
│  pivot = arr[len(arr) // 2]                                                                                     │
│     left = [x for x in arr if x < pivot]                                                                        │
│     middle = [x for x in arr if x == pivot]                                                                     │
│     right = [x for x in arr if x > pivot]                                                                       │
│     return func(left) + middle + func(right)                                                                    │
│ </                                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                     merge_quick (flipped) · SRC=quick_sort -> DST=merge_sort · I3 last-onwards · span=name+signature · V×4 · α=1.0 · pad · I2=off                                      
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pos ┃ from end ┃ token   ┃ clean SRC                                              ┃ DST                                                    ┃ patched DST                                             ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  61 │      -17 │ ' func' │ I'm sorting an array using the quicksort algorithm.    │ I'm sorting an array using the merge sort algorithm.   │ I'm sorting an array using the merge sort algorithm.    │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┤
│  76 │       -2 │ '\n'    │ I'm sorting the array by recursively partitioning it   │ I'm not executing any code here; I'm just preparing to │ I'm not executing any code here; I'm just preparing to  │
│     │          │         │ into sub-arrays and then reassembling them in order.   │ divide and conquer by recursively sorting.             │ divide and conquer by recursively sorting.              │
│     │          │         │ Here's a quick breakdown:                              │                                                        │                                                         │
│     │          │         │                                                        │ Here's a brief implementation of the merge sort        │ Here's a brief implementation of the merge sort         │
│     │          │         │ ```python                                              │ algorithm:                                             │ algorithm:                                              │
│     │          │         │ def quicksort(arr):                                    │                                                        │                                                         │
│     │          │         │     if len(arr) <= 1:                                  │ ```python                                              │ ```python                                               │
│     │          │         │         return                                         │ def merge_sort(arr):                                   │ def merge_sort(arr):                                    │
│     │          │         │                                                        │     if len(arr) > 1:                                   │     if len(arr) > 1:                                    │
├─────┼──────────┼─────────┼────────────────────────────────────────────────────────┼────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────┤
│  77 │       -1 │ '   '   │ I'm selecting a pivot element and partitioning the     │ I'm not executing any code; I'm just preparing to      │ I'm sorting the array by recursively partitioning it    │
│     │          │         │ array into two sub-arrays: elements less than the      │ divide the task into smaller parts and merge them back │ into sub-arrays, then reassembling them in order.       │
│     │          │         │ pivot and elements greater than the pivot, then        │ together. No action is taken at a single point; the    │                                                         │
│     │          │         │ recursively sorting the sub-arrays.                    │ process involves recursively splitting the list into   │                                       

## Other pairs / directions (optional — each pair×direction is 3 generations + 3 readout batches)

In [7]:
# for key in PAIRS:
#     for flip in (False, True):
#         b = build(key, flip)
#         show(b, run(b))